Creating a new file store dynamic data, put timestamp, eye data position, GI, choice, cor_wro in a new csv file.

In [1]:
# put timestamp, deg_x, deg_y, GI, choice, correct in new csv file

import os
import pandas as pd
import numpy as np

# ========== Parameters ==========
gaze_folder = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\gaze_processed"
output_folder = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data"
eye_to_screen_cm = 57  # Viewing distance from eyes to screen

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

def cm_to_deg(x_cm, y_cm, distance_cm=eye_to_screen_cm):
    x_deg = 2 * np.arctan(x_cm / (2 * distance_cm)) * 180 / np.pi
    y_deg = 2 * np.arctan(y_cm / (2 * distance_cm)) * 180 / np.pi
    return x_deg, y_deg

for filename in os.listdir(gaze_folder):
    if not filename.endswith(".csv"):
        continue

    try:
        filepath = os.path.join(gaze_folder, filename)
        df = pd.read_csv(filepath)

        # Basic columns
        time = df['time']
        gi = df['stimulus_index']
        choice = df['choice']

        # Convert gaze to degrees
        deg_x, deg_y = cm_to_deg(df['position_x'], df['position_y'])

        # Compute correctness
        gi_val = gi.iloc[0]
        choice_val = choice.iloc[0]

        if gi_val == 0:
            correct = 1
        else:
            correct = int(np.sign(gi_val) == np.sign(choice_val))

        correct_col = [correct] * len(df)

        # Output DataFrame
        new_df = pd.DataFrame({
            'time': time,
            'deg_x': deg_x,
            'deg_y': deg_y,
            'GI': gi,
            'choice': choice,
            'correct': correct_col
        })

        output_path = os.path.join(output_folder, filename)
        new_df.to_csv(output_path, index=False)

    except Exception as e:
        print(f"[ERROR] Failed to process {filename}: {e}")


Putting events into dynamic data files.

In [2]:
# put fixation information with timestamp in dynamic data

import pandas as pd
import os

# ========== 路径设置 ==========
dynamic_data_folder = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data"
events_folder = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\events_processed"

# ========== 遍历所有 dynamic trial 文件 ==========
for filename in os.listdir(dynamic_data_folder):
    if not filename.endswith(".csv"):
        continue

    dynamic_path = os.path.join(dynamic_data_folder, filename)
    events_path = os.path.join(events_folder, filename)

    # 若 events 文件不存在，跳过该 trial
    if not os.path.exists(events_path):
        print(f"[Skipped] No event file for {filename}")
        continue

    # 加载数据
    try:
        df = pd.read_csv(dynamic_path)
        events = pd.read_csv(events_path)
    except Exception as e:
        print(f"[Error] Reading {filename}: {e}")
        continue

    # 添加列：默认非fixation
    df['is_fixation'] = 0

    # 标记 fixation 区间为1
    for i, row in events.iterrows():
        if row['name'].lower() == 'fixation':
            onset = row['onset']
            offset = row['offset']
            df.loc[(df['time'] >= onset) & (df['time'] <= offset), 'is_fixation'] = 1

    # 保存
    try:
        df.to_csv(dynamic_path, index=False)
        print(f"[Processed] {filename}")
    except Exception as e:
        print(f"[Error] Saving {filename}: {e}")


[Processed] trial_111_0.csv
[Processed] trial_111_1.csv
[Processed] trial_111_10.csv
[Processed] trial_111_100.csv
[Processed] trial_111_101.csv
[Processed] trial_111_102.csv
[Processed] trial_111_103.csv
[Processed] trial_111_104.csv
[Processed] trial_111_105.csv
[Processed] trial_111_106.csv
[Processed] trial_111_107.csv
[Processed] trial_111_108.csv
[Processed] trial_111_109.csv
[Processed] trial_111_11.csv
[Processed] trial_111_110.csv
[Processed] trial_111_111.csv
[Processed] trial_111_112.csv
[Processed] trial_111_113.csv
[Processed] trial_111_114.csv
[Processed] trial_111_115.csv
[Processed] trial_111_116.csv
[Processed] trial_111_117.csv
[Processed] trial_111_118.csv
[Processed] trial_111_119.csv
[Processed] trial_111_12.csv
[Processed] trial_111_120.csv
[Processed] trial_111_121.csv
[Processed] trial_111_122.csv
[Processed] trial_111_123.csv
[Processed] trial_111_124.csv
[Processed] trial_111_125.csv
[Processed] trial_111_126.csv
[Processed] trial_111_127.csv
[Processed] trial

Putting point light walker information of each frame into new csv file.

In [3]:
# put PLW light points information with timestamp in dynamic data, time cost 1 sec per trial, may be long.

import os
import pandas as pd
import numpy as np

# ========== Parameters ==========
eye_to_screen_cm = 57  # Distance from eye to screen
n_dots = 15  # Number of dots per frame
deg_convert = lambda x_cm: 2 * np.arctan(x_cm / (2 * eye_to_screen_cm)) * (180 / np.pi)

# ========== Paths ==========
gaze_dir = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data"
frame_dir = r"Z:\BioMotionAnlyze\analyze\data\meta data\exp 202504\subtrialInfo\frame"

# ========== Process Each File ==========
for filename in os.listdir(gaze_dir):
    if not filename.endswith(".csv"):
        continue

    gaze_path = os.path.join(gaze_dir, filename)
    frame_path = os.path.join(frame_dir, filename)

    if not os.path.exists(frame_path):
        print(f"Missing frame file for: {filename}")
        continue

    # Load gaze and frame data
    gaze_df = pd.read_csv(gaze_path)
    frame_df = pd.read_csv(frame_path)

    total_rows = len(gaze_df)

    # Convert cm to deg for all dots
    for i in range(1, n_dots + 1):
        frame_df[f"x{i}_deg"] = deg_convert(frame_df[f"x{i}"])
        frame_df[f"y{i}_deg"] = deg_convert(frame_df[f"y{i}"])

    # Initialize new columns in gaze_df
    for i in range(1, n_dots + 1):
        gaze_df[f"x{i}_deg"] = np.nan
        gaze_df[f"y{i}_deg"] = np.nan

    # Get frame times
    frame_times = frame_df["frame_time"].astype(int).tolist()

    # Fill values between frames using frame-wise values
    for idx in range(len(frame_times)):
        start = frame_times[idx]
        end = frame_times[idx + 1] if idx + 1 < len(frame_times) else total_rows

        for i in range(1, n_dots + 1):
            x_val = frame_df.at[idx, f"x{i}_deg"]
            y_val = frame_df.at[idx, f"y{i}_deg"]
            gaze_df.loc[start:end - 1, f"x{i}_deg"] = x_val
            gaze_df.loc[start:end - 1, f"y{i}_deg"] = y_val

    # Save result
    gaze_df.to_csv(gaze_path, index=False)


Sortting the dynamic data by cor_wro

In [4]:
import os
import re
import shutil
import pandas as pd
from collections import Counter

"""
Classify trial CSVs into "correct" and "wrong" folders based on the `correct` column.

Requirements from user:
- Input folder contains CSV files named like:
    - trial_<subject_or_any_string>_<trial_id>.csv
  e.g., trial_11_0.csv, trial_ABC_42.csv, trial_xyz_trial-3.csv
- Each CSV contains an entire trial. Column `correct` marks trial outcome (1 = correct, 0 or -1 = wrong).
- Move/copy files to two output folders:
    - *_cor for correct trials
    - *_wro for wrong trials
- Two modes:
    - "all": process every file in the folder
    - "single": only process files whose middle identifier matches `target_id` (subject id or any string between the first and last underscore)

Edit the PARAMETERS block below to use.
"""

# ========== PARAMETERS (EDIT HERE) ==========
input_folder = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data"
output_folder_correct = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data_cor"
output_folder_wrong = r"Z:\BioMotionAnlyze\analyze\data\pymovement data\exp 202504\dynamic data_wro"

# Mode: "all" or "single"
mode = "all"  # change to "single" when you want to filter by one subject/identifier
# When mode == "single", set the identifier to match the middle part of the filename
# For example, for files like trial_11_0.csv, target_id = "11"; for trial_ABC_5.csv, target_id = "ABC"
target_id = "11"

# Copy or move? If True, files will be MOVED to the destination; if False, they will be COPIED
move_files = False

# ============================================

# Ensure output folders exist
os.makedirs(output_folder_correct, exist_ok=True)
os.makedirs(output_folder_wrong, exist_ok=True)

# Compiled regex to capture: trial_<ID>_<TRIAL>.csv (case-insensitive)
# It is forgiving for extra underscores in the tail section.
FILE_RE = re.compile(r"^trial_(?P<mid>.+)_(?P<trial>\d+)\.csv$", re.IGNORECASE)


def pick_trial_label(series):
    """Determine trial correctness from a pandas Series `correct`.
    Logic:
      1) Drop NaNs; map any positive value to 1, non-positive to 0.
      2) If unique value is single -> use it.
      3) If mixed -> use majority vote, but warn in return tuple.
    Returns (label, note) where label is 1 for correct, 0 for wrong.
    """
    vals = series.dropna().astype(float)
    if vals.empty:
        # Default to wrong if column missing/empty
        return 0, "empty-correct-column"
    mapped = (vals > 0).astype(int)
    uniq = mapped.unique()
    if len(uniq) == 1:
        return int(uniq[0]), "ok"
    # majority vote
    c = Counter(mapped)
    label = 1 if c[1] >= c[0] else 0
    return label, f"mixed({c[1]} ones, {c[0]} zeros)"


def should_take(mid: str) -> bool:
    if mode.lower() == "all":
        return True
    return str(mid) == str(target_id)


def handle_one(csv_path: str, mid: str):
    try:
        df = pd.read_csv(csv_path)
        if "correct" not in df.columns:
            # If column name is accidentally capitalized or spaced, try fuzzy alternatives
            candidates = [c for c in df.columns if c.strip().lower() == "correct"]
            if not candidates:
                raise ValueError("No 'correct' column found")
            correct_col = candidates[0]
        else:
            correct_col = "correct"
        label, note = pick_trial_label(df[correct_col])
        dest_root = output_folder_correct if label == 1 else output_folder_wrong
        dest_path = os.path.join(dest_root, os.path.basename(csv_path))
        if move_files:
            shutil.move(csv_path, dest_path)
        else:
            shutil.copy2(csv_path, dest_path)
        print(f"[OK] {'MOVED' if move_files else 'COPIED'} -> {'COR' if label==1 else 'WRO'} | {os.path.basename(csv_path)} | id={mid} | {note}")
    except Exception as e:
        print(f"[ERROR] {os.path.basename(csv_path)} | {e}")


def main():
    files = [f for f in os.listdir(input_folder) if f.lower().endswith('.csv')]
    if not files:
        print("No CSV files found.")
        return

    total = 0
    taken = 0
    for fname in files:
        m = FILE_RE.match(fname)
        if not m:
            # Skip non-matching names quietly but log once
            print(f"[SKIP] Name does not match pattern 'trial_<ID>_<TRIAL>.csv': {fname}")
            continue
        mid = m.group('mid')
        total += 1
        if not should_take(mid):
            continue
        taken += 1
        handle_one(os.path.join(input_folder, fname), mid)

    print(f"\nDone. Matched pattern files: {total}. Processed (mode={mode}): {taken}.")


if __name__ == "__main__":
    main()


[OK] COPIED -> COR | trial_111_0.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_1.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_10.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_100.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_101.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_102.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_103.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_104.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_105.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_106.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_107.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_108.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_109.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_11.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_110.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_111.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_112.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_113.csv | id=111 | ok
[OK] COPIED -> COR | trial_111_114.csv | id=111 | ok